# Colab 30 — training-size ablation + generalization ladder

Two supporting experiments, one harness (reuses the colab24f/28/29 recipe verbatim):

**A. Training-size ablation** — *why 30k pairs?* Train the AA encoder across
`N ∈ {1k, 3k, 10k, 30k, 100k}`, **3 seeds each**, **EPOCHS fixed at 30**, and plot performance vs N so
the choice is defensible ("plateaus by N≈X; 30k sits on the flat"). Y-axis = **Spearman on real CATH
AA + Spearman on a held-out synthetic set** (primary), with **MAP@10 on real AA** as a second panel.
*Caveat (stated on the slide): with epochs fixed, larger N also means more gradient steps — this is the
realistic "same schedule, more data" curve, not a compute-controlled one.*

**B. Generalization ladder** — *test at inference on the training distribution.* Take the N=30k AA
encoder and score it on **synthetic in-distribution → real AA → SS → 3Di**. Synthetic is the natural
**ceiling** (same generator, full normLev range); the drop across the ladder is the transfer cost. This
is also the one setting that exercises the *full* high-similarity range, which real CATH AA can't
(only ~6 natural high-sim AA pairs).

*Ground truth = exact normLev only. Locked design: `memory/presentation_redo_locks.md`.*

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# ---- ablation config (locked 2026-07-11) ----
N_GRID  = [1_000, 3_000, 10_000, 30_000, 100_000]
SEEDS   = [0, 1, 2]
EPOCHS  = 30
N_SYNTH = 6_000        # held-out synthetic eval pairs (fixed across all runs)
STRAT_PER_BIN = 400
STRAT_CAND    = 200_000
LADDER_N = 30_000      # which N the generalization ladder uses

## 2. Constants + helpers + architecture (colab29 recipe, verbatim)

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)
def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L
def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)
def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)
def rand_seq(abc, rng): L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))
def bin_idx(x, band_low): return 0 if x < band_low else (1 if x < BAND_HIGH else 2)

class Enc(nn.Module):
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)
class Clf(nn.Module):
    def __init__(s):
        super().__init__(); s.encoder = Enc()
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(0.01), nn.Linear(64, 3))
    def forward(s, a, b): return s.head(torch.abs(s.encoder(a) - s.encoder(b)))
class DS(Dataset):
    def __init__(s, pp, bl): s.p = pp; s.bl = bl
    def __len__(s): return len(s.p)
    def __getitem__(s, i): a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), bin_idx(l, s.bl)

def train_snn_n(alphabet, band_low, n_train, seed, label):
    # identical recipe to colab29.train_snn, but N and seed are explicit and EPOCHS is fixed.
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    pairs = []
    while len(pairs) < n_train:
        sd = rand_seq(alphabet, rng); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, alphabet, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    dl = DataLoader(DS(pairs, band_low), batch_size=BS, shuffle=True)
    model = Clf().to(device); opt = torch.optim.Adam(model.parameters(), 1e-3); crit = nn.CrossEntropyLoss()
    model.train()
    for ep in range(1, EPOCHS+1):
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = crit(model(a, b), y); opt.zero_grad(); loss.backward(); opt.step()
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); return model

## 3. Real pools (AA/SS/3Di) + fixed eval sets (built once, shared across all runs)

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')
RESCUED = {'4z0mC02', '3qkaE02'}
def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])            if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])            if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
FEEDS = ['AA', 'SS', '3Di']
for f in FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

In [ ]:
# --- exhaustive de-hubbed oracle (AA only: needed for the ablation MAP@10 curve) ---
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs)
    T_high = {}; pos_pairs = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos_pairs.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos_pairs)
print('Building AA oracle (for MAP@10 ablation panel)...')
ORACLE = {'AA': build_oracle('AA')}
print(f'  AA queries@0.70 = {len(ORACLE["AA"]["T_high"])}, high-sim pos pairs = {len(ORACLE["AA"]["pos_pairs"])}')

In [ ]:
# --- shared stratified pair set per real feed (per feed's own normLev), colab29 logic ---
def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if feed in ORACLE and ORACLE[feed]['pos_pairs']:
        parr = np.array(ORACLE[feed]['pos_pairs'], dtype=float)   # (P,3): i, j, normLev — concat once
        a = np.concatenate([a, parr[:, 0].astype(np.int64)])
        b = np.concatenate([b, parr[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, parr[:, 2]])
    edges = np.linspace(0.0, 1.0, 11); bins = np.clip(np.digitize(nl, edges) - 1, 0, 9)
    ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size == 0: continue
        take = rng.permutation(idx)[:STRAT_PER_BIN]
        ai.append(a[take]); aj.append(b[take]); av.append(nl[take])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64),
               nl=np.concatenate(av))
strat_rng = np.random.default_rng(999)
STRAT = {f: build_strat_pairs(f, strat_rng) for f in FEEDS}
for f in FEEDS: print(f'  strat pairs[{f}] = {len(STRAT[f]["nl"]):>5}  (max normLev {STRAT[f]["nl"].max():.2f})')

In [ ]:
# --- held-out synthetic eval pairs (same generator as training, disjoint rng, fixed for all runs) ---
def build_synth_pairs(m, seed=12345):
    rng = np.random.default_rng(seed); A, B, Y = [], [], []
    while len(Y) < m:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd); t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: A.append(sd); B.append(o); Y.append(norm_lev(sd, o))
    return A, B, np.array(Y)
SYN_A, SYN_B, SYN_Y = build_synth_pairs(N_SYNTH)
print(f'  synthetic eval pairs = {len(SYN_Y)}  (normLev range {SYN_Y.min():.2f}-{SYN_Y.max():.2f}, '
      f'{(SYN_Y>=BAND_HIGH).sum()} high-sim)')

## 4. Metric helpers (embed + Spearman + MAP@10 + AUROC)

In [ ]:
@torch.no_grad()
def embed_seqs(model, seqs, bs=256):
    out = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def spearman_real(model, feed):
    E = embed_seqs(model, POOL_SEQ[feed]); P = STRAT[feed]
    s = np.sum(E[P['i']]*E[P['j']], axis=1); return spearmanr(s, P['nl']).correlation

def auroc_real(model, feed):
    E = embed_seqs(model, POOL_SEQ[feed]); P = STRAT[feed]
    s = np.sum(E[P['i']]*E[P['j']], axis=1); y = (P['nl'] >= BAND_HIGH).astype(int)
    return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

def spearman_synth(model):
    Ea = embed_seqs(model, SYN_A); Eb = embed_seqs(model, SYN_B)
    return spearmanr(np.sum(Ea*Eb, axis=1), SYN_Y).correlation

def auroc_synth(model):
    Ea = embed_seqs(model, SYN_A); Eb = embed_seqs(model, SYN_B)
    s = np.sum(Ea*Eb, axis=1); y = (SYN_Y >= BAND_HIGH).astype(int)
    return roc_auc_score(y, s)

def _ap(order, ts, k=10):
    nt = len(ts)
    if nt == 0: return np.nan
    hits = 0; ap = 0.0
    for r, o in enumerate(order[:k], 1):
        if o in ts: hits += 1; ap += hits/r
    return ap/min(nt, k)
def map10_real(model, feed):
    E = embed_seqs(model, POOL_SEQ[feed]); T = ORACLE[feed]['T_high']; aps = []
    for qi, arr in T.items():
        ts = set(arr.tolist()); s = E @ E[qi]; s[qi] = -np.inf
        aps.append(_ap(np.argsort(-s), ts, 10))
    return float(np.nanmean(aps)) if aps else np.nan

## 5. Experiment A — training-size ablation (N × seed, EPOCHS fixed)

In [ ]:
rows = []
t0 = time.perf_counter()
for N in N_GRID:
    for seed in SEEDS:
        model = train_snn_n(AA_ALPHABET, BAND_LOW_AA, N, seed, f'N={N},s={seed}')
        rows.append(dict(N=N, seed=seed,
                         spearman_aa=spearman_real(model, 'AA'),
                         spearman_syn=spearman_synth(model),
                         map10_aa=map10_real(model, 'AA')))
        print(f'  N={N:>7} seed={seed}  rho_AA={rows[-1]["spearman_aa"]:.3f}  '
              f'rho_syn={rows[-1]["spearman_syn"]:.3f}  MAP@10_AA={rows[-1]["map10_aa"]:.3f}')
abl = pd.DataFrame(rows)
agg = abl.groupby('N').agg(['mean', 'std'])
print('\n' + '='*72); print('ABLATION (mean ± std over seeds)'); print('='*72)
print(agg.round(3).to_string())
abl.to_csv('colab30_ablation.csv', index=False)

In [ ]:
# --- plateau / knee: smallest N whose mean Spearman(real AA) reaches >=99% of the max ---
m = abl.groupby('N')['spearman_aa'].mean()
knee = int(m.index[(m >= 0.99*m.max())][0])
print(f'Spearman(real AA) reaches >=99% of its max ({m.max():.3f}) at N={knee:,}.')
print(f'Chosen N=30,000 sits {"on/after" if 30000>=knee else "before"} the knee.')

In [ ]:
import matplotlib.pyplot as plt
def despine(ax):
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
def band(ax, col, color, label):
    g = abl.groupby('N')[col]; mean = g.mean(); std = g.std().fillna(0)
    ax.plot(mean.index, mean.values, 'o-', color=color, label=label)
    ax.fill_between(mean.index, mean.values-std.values, mean.values+std.values, color=color, alpha=0.18)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
band(ax[0], 'spearman_aa', '#1f77b4', 'real CATH AA')
band(ax[0], 'spearman_syn', '#ff7f0e', 'synthetic (in-distribution)')
ax[0].set_ylabel('Spearman ρ(sim, normLev)'); ax[0].set_title('Spearman vs training size (3 seeds)')
band(ax[1], 'map10_aa', '#1f77b4', 'real CATH AA')
ax[1].set_ylabel('MAP@10 (real AA, full-pool)'); ax[1].set_title('Retrieval vs training size (3 seeds)')
for a in ax:
    a.set_xscale('log'); a.set_xlabel('training pairs N'); a.axvline(30000, ls=':', color='grey')
    a.annotate('N=30k', (30000, a.get_ylim()[0]), fontsize=9, color='grey', rotation=90, va='bottom')
    a.legend(); despine(a)
plt.tight_layout(); plt.savefig('colab30_ablation.png', dpi=150, bbox_inches='tight'); plt.show()

## 6. Experiment B — generalization ladder (N=30k AA encoder across distributions)

One model, four distributions: synthetic in-distribution (ceiling) → real AA → SS → 3Di. Uses the AA
encoder (transfer story). SS/3Di strings tokenise through the AA vocabulary (both use AA-letter
alphabets), exactly as in colab29.

In [ ]:
ladder_model = train_snn_n(AA_ALPHABET, BAND_LOW_AA, LADDER_N, SEEDS[0], f'ladder N={LADDER_N}')
ladder = []
ladder.append(dict(dist='synthetic', rho=spearman_synth(ladder_model), auroc=auroc_synth(ladder_model)))
for f in FEEDS:
    ladder.append(dict(dist=f, rho=spearman_real(ladder_model, f), auroc=auroc_real(ladder_model, f)))
L = pd.DataFrame(ladder)
print('='*56); print('GENERALIZATION LADDER (AA encoder, N=30k)'); print('='*56)
print(L.round(3).to_string(index=False))
L.to_csv('colab30_ladder.csv', index=False)

In [ ]:
order = ['synthetic', 'AA', 'SS', '3Di']; x = np.arange(len(order)); w = 0.38
Lx = L.set_index('dist').reindex(order)
fig, ax = plt.subplots(figsize=(8.6, 5))
b1 = ax.bar(x - w/2, Lx['rho'].values, w, label='Spearman ρ', color='#1f77b4')
b2 = ax.bar(x + w/2, Lx['auroc'].values, w, label='AUROC (≥0.70)', color='#2ca02c')
ax.bar_label(b1, fmt='%.2f', fontsize=9); ax.bar_label(b2, fmt='%.2f', fontsize=9)
ax.axhline(0.5, ls='--', color='grey', lw=1)
ax.set_xticks(x); ax.set_xticklabels(['synthetic\n(ceiling)', 'real AA', 'SS', '3Di'])
ax.set_ylim(0, 1.08); ax.set_ylabel('score')
ax.set_title('Generalization ladder — one AA encoder, in-distribution → real → cross-alphabet')
ax.legend(); despine(ax)
plt.tight_layout(); plt.savefig('colab30_ladder.png', dpi=150, bbox_inches='tight'); plt.show()

## 7. How to read this notebook

**Experiment A (the "why 30k?" defense):** §5 prints the mean±std curve; §5b prints the **knee**
(smallest N reaching ≥99% of the max Spearman on real AA). Say on the slide: *"performance plateaus by
N≈{knee}; 30k sits on the flat, so more data buys little."* The synthetic curve should saturate too and
sit above the real-AA curve (it's the in-distribution ceiling). **Honesty line for the slide:** epochs
are fixed at 30, so larger N also means more gradient steps — this is the realistic "same schedule,
more data" curve, not a compute-controlled one (that variant is a backup/outlook).

**Experiment B (the "test on the training distribution" check):** §6 is the ladder. Expect
`synthetic ≥ real AA ≥ SS ≈ 3Di`. The synthetic bar is the **ceiling** — it's the only column that
exercises the full high-similarity range (real CATH AA has ~6 natural high-sim pairs). The gap from
synthetic to real AA is the distribution-shift cost; the further gap to SS/3Di is the cross-alphabet
transfer cost (the prominent-secondary result). If synthetic is *not* clearly best, that's a red flag
worth investigating before the talk.

Outputs: `colab30_ablation.csv/png`, `colab30_ladder.csv/png`.